In [78]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler


In [79]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [80]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [81]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [82]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [83]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [84]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [85]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [86]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [87]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [88]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [89]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [90]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [91]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]


# Selecting features in the DataFrame
X_rent = X[selected_features]

In [92]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [93]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [94]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [95]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [96]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,1.075042,0.081641,-0.098422,-0.562251
1,0.242767,-0.395928,-1.209119,-0.562251
2,-1.461057,0.208583,0.280542,-0.202838
3,0.078112,-0.949323,-1.147026,-0.562251
4,-2.274305,1.555538,-0.568449,1.514080
...,...,...,...,...
134,0.827589,-0.217633,-0.671191,-0.562251
135,0.578577,1.572043,-0.163918,-0.562251
136,-0.329498,-0.648232,-0.445412,-0.562251
137,0.595927,1.711090,0.143137,-0.562251


In [97]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [98]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,-0.133381,0.800544,0.405980,-0.288893
1,-0.108854,-1.860610,-0.118774,1.182201
2,-0.704414,-0.754955,0.048037,0.033974
3,-1.307469,-1.134568,0.158761,-0.562251
4,-0.615466,-0.128772,0.685437,-0.562251
...,...,...,...,...
94,-0.085581,0.263605,-0.598102,-0.562251
95,-0.599161,-0.859260,0.472444,-0.562251
96,-0.425762,-1.598975,0.432435,-0.562251
97,0.637891,-0.499761,0.081495,-0.562251


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [99]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 22:05:26,946] A new study created in memory with name: no-name-1bf8c919-463d-45bb-89a9-fb488593abcf


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-14 22:05:27,292] A new study created in memory with name: no-name-ed177a89-2bdb-4577-a351-fa40d1691fb0


Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:27,222] Trial 0 finished with value: 0.6885812252956434 and parameters: {}. Best is trial 0 with value: 0.6885812252956434.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6885812252956434], datetime_start=datetime.datetime(2024, 4, 14, 22, 5, 27, 15861), datetime_complete=datetime.datetime(2024, 4, 14, 22, 5, 27, 222679), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6885812252956434


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24445113125013965
Fold 2 IBS: 0.19124878012978022
Fold 3 IBS: 0.20665297767216081
Fold 4 IBS: 0.19438765965098836
Fold 5 IBS: 0.197759489882654
[I 2024-04-14 22:05:27,518] Trial 0 finished with value: 0.20690000771714462 and parameters: {}. Best is trial 0 with value: 0.20690000771714462.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20690000771714462], datetime_start=datetime.datetime(2024, 4, 14, 22, 5, 27, 354043), datetime_complete=datetime.datetime(2024, 4, 14, 22, 5, 27, 517901), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20690000771714462


In [100]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [101]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.689
train_ibs:  0.207


#### Test

In [102]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [103]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.532
IBS score: 0.29


In [104]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [105]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [106]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 22:05:27,756] A new study created in memory with name: no-name-1f4e87a3-d196-47f7-a3cc-6037e62d6369


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7422480620155039
Fold 3 C-index: 0.5702127659574469


[I 2024-04-14 22:05:27,939] A new study created in memory with name: no-name-ec35f23f-3d56-4b17-ada6-d8a187e899bf


Fold 4 C-index: 0.7167300380228137
Fold 5 C-index: 0.6759656652360515
[I 2024-04-14 22:05:27,928] Trial 0 finished with value: 0.6629436568439727 and parameters: {}. Best is trial 0 with value: 0.6629436568439727.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6629436568439727], datetime_start=datetime.datetime(2024, 4, 14, 22, 5, 27, 794921), datetime_complete=datetime.datetime(2024, 4, 14, 22, 5, 27, 928313), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6629436568439727


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709421972138
Fold 2 IBS: 0.23203987311750648
Fold 3 IBS: 0.22898186706001533
Fold 4 IBS: 0.24197476010091573
Fold 5 IBS: 0.2293955839675724
[I 2024-04-14 22:05:28,175] Trial 0 finished with value: 0.23592783569314624 and parameters: {}. Best is trial 0 with value: 0.23592783569314624.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783569314624], datetime_start=datetime.datetime(2024, 4, 14, 22, 5, 28, 10361), datetime_complete=datetime.datetime(2024, 4, 14, 22, 5, 28, 175129), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783569314624


In [107]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [108]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.663
train_ibs:  0.236


#### Test

In [109]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [110]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.539


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [111]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [112]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 22:05:28,377] A new study created in memory with name: no-name-d32ca149-edb0-4f69-83c2-92e30e96a087


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734


[I 2024-04-14 22:05:28,824] A new study created in memory with name: no-name-761d7a50-1aba-441a-8ce3-9aea47dbab6f


Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:28,819] Trial 0 finished with value: 0.6910259146234387 and parameters: {}. Best is trial 0 with value: 0.6910259146234387.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6910259146234387], datetime_start=datetime.datetime(2024, 4, 14, 22, 5, 28, 461041), datetime_complete=datetime.datetime(2024, 4, 14, 22, 5, 28, 819351), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6910259146234387


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24331006599898158
Fold 2 IBS: 0.1911934758131866
Fold 3 IBS: 0.20675927385724954
Fold 4 IBS: 0.19435877417580916
Fold 5 IBS: 0.198512695677751
[I 2024-04-14 22:05:29,184] Trial 0 finished with value: 0.2068268571045956 and parameters: {}. Best is trial 0 with value: 0.2068268571045956.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2068268571045956], datetime_start=datetime.datetime(2024, 4, 14, 22, 5, 28, 864532), datetime_complete=datetime.datetime(2024, 4, 14, 22, 5, 29, 184110), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2068268571045956


In [113]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [114]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.691
train_ibs:  0.207


#### Test 

In [115]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [116]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.288


In [117]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [118]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 22:05:29,502] A new study created in memory with name: no-name-b9c053af-9e49-452e-b276-c23fd78123db


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:30,071] Trial 0 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:30,347] Trial 1 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:30,632] Trial 2 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:41,087] Trial 24 finished with value: 0.6893780380446474 and parameters: {'l1_ratio': 0.15433706302991543}. Best is trial 0 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:41,509] Trial 25 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.4626599986704694}. Best is trial 0 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:41,842] Trial 26 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.3312072877416866}

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:48,685] Trial 48 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.5675456270820948}. Best is trial 0 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:48,973] Trial 49 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.7153190338760974}. Best is trial 0 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:05:49,339] Trial 50 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.5027665121167861}

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2433156114036618
Fold 2 IBS: 0.19115530450118298
Fold 3 IBS: 0.20654328851646514
Fold 4 IBS: 0.19431453572655688
Fold 5 IBS: 0.19840304293834035
[I 2024-04-14 22:06:04,582] Trial 0 finished with value: 0.20674635661724144 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.20674635661724144.
Fold 1 IBS: 0.24345534898240948
Fold 2 IBS: 0.19106167895592804
Fold 3 IBS: 0.2059890726266797
Fold 4 IBS: 0.1941873317240624
Fold 5 IBS: 0.1981025500302711
[I 2024-04-14 22:06:04,902] Trial 1 finished with value: 0.20655919646387016 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.20655919646387016.
Fold 1 IBS: 0.24352041762954318
Fold 2 IBS: 0.19104563842317818
Fold 3 IBS: 0.20588536144687355
Fold 4 IBS: 0.1941465437949834
Fold 5 IBS: 0.19803689766826307
[I 2024-04-14 22:06:05,239] Trial 2 finished with value: 0.20652697179256826 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.20652697179256

Fold 1 IBS: 0.2450682640426586
Fold 2 IBS: 0.22795487619322505
Fold 3 IBS: 0.22881012278568916
Fold 4 IBS: 0.23799365210664025
Fold 5 IBS: 0.2262129117046938
[I 2024-04-14 22:06:12,869] Trial 25 finished with value: 0.23320796536658137 and parameters: {'l1_ratio': 0.03981317955666948}. Best is trial 12 with value: 0.20640201895757088.
Fold 1 IBS: 0.24359131338350348
Fold 2 IBS: 0.19102169658035204
Fold 3 IBS: 0.20575844749744293
Fold 4 IBS: 0.19413589776762996
Fold 5 IBS: 0.19800656094777605
[I 2024-04-14 22:06:14,392] Trial 26 finished with value: 0.2065027832353409 and parameters: {'l1_ratio': 0.18783928730207888}. Best is trial 12 with value: 0.20640201895757088.
Fold 1 IBS: 0.2433607473731579
Fold 2 IBS: 0.19113811271624523
Fold 3 IBS: 0.20643511239831425
Fold 4 IBS: 0.19429297011760782
Fold 5 IBS: 0.19833925037164454
[I 2024-04-14 22:06:15,555] Trial 27 finished with value: 0.20671323859539395 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 12 with value: 0.206402018

Fold 1 IBS: 0.24334390886691287
Fold 2 IBS: 0.19113075065475335
Fold 3 IBS: 0.20639867222499295
Fold 4 IBS: 0.19428515589467615
Fold 5 IBS: 0.1982849899059779
[I 2024-04-14 22:06:28,380] Trial 50 finished with value: 0.20668869550946267 and parameters: {'l1_ratio': 0.556147470734502}. Best is trial 12 with value: 0.20640201895757088.
Fold 1 IBS: 0.2436253663181637
Fold 2 IBS: 0.19096502589656006
Fold 3 IBS: 0.2055467515574163
Fold 4 IBS: 0.19406476572289086
Fold 5 IBS: 0.19788395464949432
[I 2024-04-14 22:06:28,878] Trial 51 finished with value: 0.20641717282890504 and parameters: {'l1_ratio': 0.09816171051292913}. Best is trial 12 with value: 0.20640201895757088.
Fold 1 IBS: 0.2436224995486653
Fold 2 IBS: 0.19096369692309786
Fold 3 IBS: 0.20554085835871463
Fold 4 IBS: 0.19406317611055732
Fold 5 IBS: 0.1978838412055684
[I 2024-04-14 22:06:29,403] Trial 52 finished with value: 0.2064148144293207 and parameters: {'l1_ratio': 0.09767894737350165}. Best is trial 12 with value: 0.2064020189

Fold 2 IBS: 0.19105845803079508
Fold 3 IBS: 0.2059598496156694
Fold 4 IBS: 0.19418181145185112
Fold 5 IBS: 0.1980688102664647
[I 2024-04-14 22:06:40,208] Trial 75 finished with value: 0.20655404708534028 and parameters: {'l1_ratio': 0.2610656085136891}. Best is trial 74 with value: 0.2063935593925948.
Fold 1 IBS: 0.24354999743726324
Fold 2 IBS: 0.19102139129203694
Fold 3 IBS: 0.2058431805096221
Fold 4 IBS: 0.1941367640036499
Fold 5 IBS: 0.19800866175433174
[I 2024-04-14 22:06:40,614] Trial 76 finished with value: 0.2065119989993808 and parameters: {'l1_ratio': 0.20331332808738323}. Best is trial 74 with value: 0.2063935593925948.
Fold 1 IBS: 0.24573621227936895
Fold 2 IBS: 0.22922850186128313
Fold 3 IBS: 0.22882001812587105
Fold 4 IBS: 0.23921619684147438
Fold 5 IBS: 0.2271907309066142
[I 2024-04-14 22:06:40,753] Trial 77 finished with value: 0.23403833200292232 and parameters: {'l1_ratio': 0.026460277308053073}. Best is trial 74 with value: 0.2063935593925948.
Fold 1 IBS: 0.2435826361

In [119]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [120]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.691
train_ibs:  0.206


#### Test

In [121]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [122]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.533


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.08232138230466424)

test_ibs:  0.288


In [123]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [124]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 22:06:48,670] A new study created in memory with name: no-name-b8a79389-2ab4-436b-a25c-d59e3fb90ba9


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.7281368821292775
Fold 5 C-index: 0.648068669527897
[I 2024-04-14 22:06:52,215] Trial 0 finished with value: 0.6977075894586338 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6977075894586338.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7276595744680852
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:06:55,565] Trial 1 finished with value: 0.7128781680014569 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 4 C-index: 0.7813688212927756
Fold 5 C-index: 0.703862660944206
[I 2024-04-14 22:07:21,762] Trial 15 finished with value: 0.7359974816455777 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 6, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 135, 'oob_score': True, 'max_samples': 0.9916077098875709, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2050785814281292, 'warm_start': True}. Best is trial 14 with value: 0.7401429490839831.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.779467680608365
Fold 5 C-index: 0.7060085836909872
[I 2024-04-14 22:07:21,941] Trial 16 finished with value: 0.7424649927687818 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 7, 'oob_score': True, 'max_samples': 0.9632036286697128, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19855793167729388, 'warm_start': True}. Best is trial

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.813953488372093
Fold 3 C-index: 0.8468085106382979
Fold 4 C-index: 0.8174904942965779
Fold 5 C-index: 0.7725321888412017
[I 2024-04-14 22:07:36,783] Trial 30 finished with value: 0.7736629125252518 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 17, 'n_estimators': 369, 'oob_score': True, 'max_samples': 0.6833587310905035, 'max_features': None, 'min_weight_fraction_leaf': 0.008532800179007469, 'warm_start': True}. Best is trial 29 with value: 0.7751733627229969.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.8136882129277566
Fold 5 C-index: 0.7639484978540773
[I 2024-04-14 22:07:39,052] Trial 31 finished with value: 0.7735002862697369 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 379, 'oob_score': True, 'max_samples': 0.6547593902381754,

...

Fold 1 IBS: 0.22612682993385486
Fold 2 IBS: 0.17223298934748948
Fold 3 IBS: 0.20782358499736456
Fold 4 IBS: 0.20181112716562463
Fold 5 IBS: 0.21350833459590543
[I 2024-04-14 22:12:30,516] Trial 32 finished with value: 0.2043005732080478 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 8, 'n_estimators': 316, 'oob_score': False, 'max_samples': 0.8753375341925101, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.2230974552568295}. Best is trial 1 with value: 0.20384699429436565.
Fold 1 IBS: 0.22177687649658975
Fold 2 IBS: 0.17630377861515312
Fold 3 IBS: 0.2122309287307206
Fold 4 IBS: 0.2017574245411477
Fold 5 IBS: 0.21470545803951144
[I 2024-04-14 22:12:35,348] Trial 33 finished with value: 0.20535489328462447 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 307, 'oob_score': False, 'max_samples': 0.8780403780133842, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.23790435513187447
Fold 2 IBS: 0.16917310562036944
Fold 3 IBS: 0.2032724536335342
Fold 4 IBS: 0.20275263571784216
Fold 5 IBS: 0.21198800508925086
[I 2024-04-14 22:13:08,203] Trial 48 finished with value: 0.20501811103857426 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 14, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 369, 'oob_score': False, 'max_samples': 0.862646612475437, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18882619674434536}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21643739431200149
Fold 2 IBS: 0.18002210069568697
Fold 3 IBS: 0.20963559196115233
Fold 4 IBS: 0.20201711337877098
Fold 5 IBS: 0.21711926927169345
[I 2024-04-14 22:13:08,991] Trial 49 finished with value: 0.20504629392386103 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 12, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 68, 'oob_score': False, 'max_samples': 0.712240935894755, 'max_features': 'sqrt', 'min_weight_fraction

Fold 5 IBS: 0.21281753768850425
[I 2024-04-14 22:13:27,746] Trial 63 finished with value: 0.20478734721351177 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 16, 'min_samples_leaf': 12, 'max_depth': 6, 'n_estimators': 145, 'oob_score': False, 'max_samples': 0.5771231216606082, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.05763095148942208}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21904071817956408
Fold 2 IBS: 0.17322770907939164
Fold 3 IBS: 0.21071337338690455
Fold 4 IBS: 0.20198953222034577
Fold 5 IBS: 0.21531568867352077
[I 2024-04-14 22:13:29,160] Trial 64 finished with value: 0.20405740430794536 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 2, 'n_estimators': 74, 'oob_score': False, 'max_samples': 0.6331344922813995, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1612709153840626}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2211019166635484
Fold 2 IBS: 0.175

Fold 2 IBS: 0.17356537149199874
Fold 3 IBS: 0.19348965127184473
Fold 4 IBS: 0.20729301862706373
Fold 5 IBS: 0.21518309439089522
[I 2024-04-14 22:13:47,424] Trial 79 finished with value: 0.2023912644271136 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 6, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.7940170614934956, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19696931079449848}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2449767955817733
Fold 2 IBS: 0.172095460075217
Fold 3 IBS: 0.21463433607213933
Fold 4 IBS: 0.21510140082957746
Fold 5 IBS: 0.21527266884222518
[I 2024-04-14 22:13:47,885] Trial 80 finished with value: 0.21241613228018644 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 6, 'n_estimators': 28, 'oob_score': False, 'max_samples': 0.8170871700419534, 'max_features': None, 'min_weight_fraction_leaf': 0.21125680575069572}. Best is

Fold 1 IBS: 0.26932513184118945
Fold 2 IBS: 0.1697323509215535
Fold 3 IBS: 0.20905255797983618
Fold 4 IBS: 0.2258672184082545
Fold 5 IBS: 0.2152407835992728
[I 2024-04-14 22:13:57,655] Trial 95 finished with value: 0.21784360855002127 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.8547756485942715, 'max_features': None, 'min_weight_fraction_leaf': 0.17594983744388534}. Best is trial 86 with value: 0.20204409612264693.
Fold 1 IBS: 0.22065985212578162
Fold 2 IBS: 0.1724043068676393
Fold 3 IBS: 0.212876495767763
Fold 4 IBS: 0.20196603399913246
Fold 5 IBS: 0.21590968295775706
[I 2024-04-14 22:13:58,542] Trial 96 finished with value: 0.2047632743436147 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 7, 'min_samples_leaf': 15, 'max_depth': 13, 'n_estimators': 61, 'oob_score': False, 'max_samples': 0.7614906038154946, 'max_features': 'log2', 'min_weight_fraction_leaf': 

In [125]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [126]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.83
train_ibs:  0.202


#### Test

In [127]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [128]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=18, max_leaf_nodes=11,
                     max_samples=0.9059465751594485, min_samples_leaf=1,
                     min_weight_fraction_leaf=0.029068528038335394,
                     n_estimators=318, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.544


RandomSurvivalForest(max_depth=13, max_features='log2', max_leaf_nodes=11,
                     max_samples=0.7105780352803006, min_samples_leaf=11,
                     min_samples_split=13,
                     min_weight_fraction_leaf=0.2021626029496649,
                     n_estimators=31, random_state=123)

test_ibs:  0.247


In [129]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [130]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 22:14:03,303] A new study created in memory with name: no-name-011f1d51-0a43-4b35-b959-37a048c17fc5


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-14 22:14:04,114] Trial 0 finished with value: 0.7329375617815092 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7329375617815092.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:14:05,935] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:14:28,070] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 20, 'n_estimators': 246, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.12362405012814498, 'min_weight_fraction_leaf': 0.18949444225079395}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7604562737642585
Fold 5 C-index: 0.6909871244635193
[I 2024-04-14 22:14:28,897] Trial 17 finished with value: 0.7333650628805521 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 9, 'n_estimators': 362, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8640754037508087, 'min_weight_fraction_leaf': 0.08300984976320

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.7566539923954373
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 22:14:52,661] Trial 31 finished with value: 0.7301852441402965 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 11, 'n_estimators': 288, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8780867392507613, 'min_weight_fraction_leaf': 0.08053342835298444}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.7829787234042553
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-14 22:14:54,447] Trial 32 finished with value: 0.7270669981840107 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.2369623605519169
Fold 2 IBS: 0.2015664827514855
Fold 3 IBS: 0.2044196732107329
Fold 4 IBS: 0.2152719150958778
Fold 5 IBS: 0.2123358137500061
[I 2024-04-14 22:19:06,275] Trial 30 finished with value: 0.21411124907200385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.20923338228172916.
Fold 1 IBS: 0.23451695205118067
Fold 2 IBS: 0.19751010562426458
Fold 3 IBS: 0.2027198084019687
Fold 4 IBS: 0.21411412044401185
Fold 5 IBS: 0.2111761169753505
[I 2024-04-14 22:19:12,687] Trial 31 finished with value: 0.21200742069935527 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.923

Fold 1 IBS: 0.23247724033039832
Fold 2 IBS: 0.18748554961121114
Fold 3 IBS: 0.2090170455982855
Fold 4 IBS: 0.207742375469341
Fold 5 IBS: 0.20889347688505747
[I 2024-04-14 22:20:45,018] Trial 45 finished with value: 0.2091231375788587 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 144, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8505483502716148, 'min_weight_fraction_leaf': 0.09794633817349793}. Best is trial 45 with value: 0.2091231375788587.
Fold 1 IBS: 0.2314286160573986
Fold 2 IBS: 0.1860889745854672
Fold 3 IBS: 0.20638267011548084
Fold 4 IBS: 0.20884587453459477
Fold 5 IBS: 0.20805701088484838
[I 2024-04-14 22:20:47,490] Trial 46 finished with value: 0.20816062923555795 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 138, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.77092

Fold 1 IBS: 0.2376464824369693
Fold 2 IBS: 0.21593994734134395
Fold 3 IBS: 0.22177987321764975
Fold 4 IBS: 0.22790863488656224
Fold 5 IBS: 0.2205529164521157
[I 2024-04-14 22:21:22,972] Trial 60 finished with value: 0.22476557086692814 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 74, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6041879821929975, 'min_weight_fraction_leaf': 0.1056236625824671}. Best is trial 46 with value: 0.20816062923555795.
Fold 1 IBS: 0.23781793031064252
Fold 2 IBS: 0.1862495208476139
Fold 3 IBS: 0.20248456745784213
Fold 4 IBS: 0.21234711020159586
Fold 5 IBS: 0.2076972639565933
[I 2024-04-14 22:21:26,439] Trial 61 finished with value: 0.20931927855485752 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 197, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.71011260

Fold 1 IBS: 0.23693321208405524
Fold 2 IBS: 0.19000973925047765
Fold 3 IBS: 0.20243881773088923
Fold 4 IBS: 0.2110667431713562
Fold 5 IBS: 0.20765848231427028
[I 2024-04-14 22:21:52,287] Trial 75 finished with value: 0.20962139891020973 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 18, 'n_estimators': 104, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.49599352625156035, 'min_weight_fraction_leaf': 0.03138093454428739}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.2375162774090104
Fold 2 IBS: 0.21696007860868566
Fold 3 IBS: 0.2199964189465905
Fold 4 IBS: 0.2298197447784768
Fold 5 IBS: 0.2208032877666601
[I 2024-04-14 22:21:53,558] Trial 76 finished with value: 0.2250191615018847 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 14, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 81, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6498539

Fold 1 IBS: 0.23900229487413618
Fold 2 IBS: 0.1829989350900894
Fold 3 IBS: 0.20504008460974205
Fold 4 IBS: 0.21135329415618298
Fold 5 IBS: 0.20658610615550185
[I 2024-04-14 22:22:28,151] Trial 90 finished with value: 0.20899614297713048 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 16, 'n_estimators': 95, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7812194815989179, 'min_weight_fraction_leaf': 0.052583924158953604}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.23336269917153504
Fold 2 IBS: 0.18793910685243745
Fold 3 IBS: 0.20227191981491185
Fold 4 IBS: 0.2126322523584538
Fold 5 IBS: 0.20891055719221097
[I 2024-04-14 22:22:30,242] Trial 91 finished with value: 0.2090233070779098 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 170, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7

In [131]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [132]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.764
train_ibs:  0.207


#### Test

In [133]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [134]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=6, max_features=None, max_leaf_nodes=13,
                   max_samples=0.9529576562335005, min_samples_split=15,
                   min_weight_fraction_leaf=0.05156899271012762,
                   n_estimators=376, random_state=123, warm_start=True)

C-index score: 0.543


ExtraSurvivalTrees(max_depth=19, max_features=None, max_leaf_nodes=17,
                   max_samples=0.6638490929451707, min_samples_leaf=2,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.052914450220883674,
                   n_estimators=60, oob_score=True, random_state=123)

IBS: 0.256


In [135]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [136]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 22:22:52,774] A new study created in memory with name: no-name-9363ac8e-6336-4eff-aba2-fdafdd2a391a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:23:12,072] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:23:23,203] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:29:04,585] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:29:48,867] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:45:38,253] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 22:46:03,093] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 23:39:27,144] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 53 with value: 0.2348064399929884.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:39:45,643] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.357604

Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:42:40,086] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:42:48,981] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.001312025546225645, 'dropout_rate': 0.8046

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 23:45:36,682] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:45:45,640] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209832, 'dropout_rate': 0.625165508

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:48:25,026] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.0030671608519508686, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.27772784044341225, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 23:48:44,612] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9240076064991638, 'learning_rate': 0.036372907201929795, 'dropout_rate': 0.166

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 23:51:47,842] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.06808689488183181, 'dropout_rate': 0.25443633448028735, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 0.5386043790221791, 'min_weight_fraction_leaf': 0.03403879613376609, 'max_features': 'auto', 'min_impurity_decrease': 8.69596280226011e-07, 'validation_fraction': 0.9221483446288596, 'min_samples_split': 20, 'max_leaf_nodes': 12, 'min_samples_leaf': 16, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23301282639832266], datetime_start=datetime.datetime(2024, 4, 14, 23, 47, 15, 699823), datetime_complete=datetime.datetime(2024, 4, 14, 23, 47, 35, 889019), params={'subsample': 0.9481727373988897, 'learning_rate': 0.0202297525909

In [137]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [138]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.713
train_ibs:  0.233


#### Test

In [139]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [140]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.022737959076050005,
                                 dropout_rate=0.8285090704895912,
                                 learning_rate=0.0513322635401625, max_depth=2,
                                 max_features='sqrt', max_leaf_nodes=10,
                                 min_impurity_decrease=0.002070496973827786,
                                 min_samples_leaf=15, min_samples_split=19,
                                 min_weight_fraction_leaf=0.3101145382804477,
                                 n_estimators=485, random_state=123,
                                 subsample=0.45214538811231725,
                                 validation_fraction=0.9071319756271838)

C-index score: 0.532


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.229


In [141]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [142]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 23:51:53,347] A new study created in memory with name: no-name-ea06af8c-d35e-4141-82ab-615ba49aa515


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6609442060085837
[I 2024-04-14 23:51:53,651] Trial 0 finished with value: 0.6249682148055231 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6566523605150214
[I 2024-04-14 23:51:56,328] Trial 1 finished with value: 0.6241098457068107 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.5829787234042553
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6781115879828327
[I 2024-04-14 23:52:19,805] Trial 19 finished with value: 0.6488830367709311 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.6585125168493072.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 23:52:22,361] Trial 20 finished with value: 0.6430737491011814 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.6585125168493072.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.5702127659574469

Fold 5 C-index: 0.6824034334763949
[I 2024-04-14 23:52:44,647] Trial 37 finished with value: 0.6385093425346609 and parameters: {'subsample': 0.23671119278535047, 'dropout_rate': 0.5461106068967396, 'n_estimators': 166, 'learning_rate': 0.059207327336158105}. Best is trial 12 with value: 0.6585125168493072.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 23:52:45,814] Trial 38 finished with value: 0.6289447028736863 and parameters: {'subsample': 0.32943357291500935, 'dropout_rate': 0.9353840762852635, 'n_estimators': 340, 'learning_rate': 0.07288612365177065}. Best is trial 12 with value: 0.6585125168493072.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-14 23:52:46,603] Trial 39 finished with value: 0.62325147

...

Fold 3 IBS: 0.2756019255493613
Fold 4 IBS: 0.2426706185564074
Fold 5 IBS: 0.22283081379821795
[I 2024-04-14 23:55:24,016] Trial 19 finished with value: 0.24306929610718608 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.24550608887605468
Fold 2 IBS: 0.2133505584021398
Fold 3 IBS: 0.23294747738469943
Fold 4 IBS: 0.23024731593258208
Fold 5 IBS: 0.2123372739360029
[I 2024-04-14 23:55:24,184] Trial 20 finished with value: 0.22687774290629575 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2448259892509062
Fold 2 IBS: 0.21786672573833757
Fold 3 IBS: 0.23050391968540718
Fold 4 IBS: 0.23249100529840805
Fold 5 IBS: 0.215533050087791
[I 2024-04-14 23:55:24,344] Trial 21 finis

Fold 3 IBS: 0.2566141669263308
Fold 4 IBS: 0.23453614765450895
Fold 5 IBS: 0.2122914778844625
[I 2024-04-14 23:55:32,325] Trial 38 finished with value: 0.2333975796328732 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2943463281154152
Fold 2 IBS: 0.21435275898375683
Fold 3 IBS: 0.292776623523439
Fold 4 IBS: 0.2606234117164125
Fold 5 IBS: 0.24191548716779984
[I 2024-04-14 23:55:32,517] Trial 39 finished with value: 0.26080292190136467 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2956705905294083
Fold 2 IBS: 0.21430445848659016
Fold 3 IBS: 0.2928588854990794
Fold 4 IBS: 0.2596622123064148
Fold 5 IBS: 0.24678618384472736
[I 2024-04-14 23:55:32,876] Trial 40 finished 

Fold 5 IBS: 0.27461860801929316
[I 2024-04-14 23:55:37,874] Trial 57 finished with value: 0.29238736666151666 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2524958013314946
Fold 2 IBS: 0.20467957580837548
Fold 3 IBS: 0.24466604990129345
Fold 4 IBS: 0.22951388023914565
Fold 5 IBS: 0.21232254648299242
[I 2024-04-14 23:55:38,023] Trial 58 finished with value: 0.22873557075266032 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2450766728615442
Fold 2 IBS: 0.21570571438138603
Fold 3 IBS: 0.23170031253784298
Fold 4 IBS: 0.23100841297079955
Fold 5 IBS: 0.21387190006582926
[I 2024-04-14 23:55:38,122] Trial 59 finished with value: 0.2274726025634804 and parameters: {'subsample

Fold 1 IBS: 0.2506811440240583
Fold 2 IBS: 0.2053539697592853
Fold 3 IBS: 0.2432701864482772
Fold 4 IBS: 0.2284683602008423
Fold 5 IBS: 0.21006566848816424
[I 2024-04-14 23:55:46,554] Trial 77 finished with value: 0.22756786578412544 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.24728609182637104
Fold 2 IBS: 0.20948456237788454
Fold 3 IBS: 0.2367879689804507
Fold 4 IBS: 0.22900068533758589
Fold 5 IBS: 0.21042389725248523
[I 2024-04-14 23:55:46,739] Trial 78 finished with value: 0.2265966411549555 and parameters: {'subsample': 0.5939500427136554, 'dropout_rate': 0.7406477151463077, 'n_estimators': 73, 'learning_rate': 0.015420072941943445}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2491522359439618
Fold 2 IBS: 0.20720785646863638
Fold 3 IBS: 0.23970965460068624
Fold 4 IBS: 0.22880108027481874
Fold 5 IB

Fold 2 IBS: 0.26256077094216035
Fold 3 IBS: 0.35927296600690356
Fold 4 IBS: 0.2978922662575483
Fold 5 IBS: 0.3041795583962892
[I 2024-04-14 23:55:51,509] Trial 96 finished with value: 0.31535726936063474 and parameters: {'subsample': 0.41313939490511475, 'dropout_rate': 0.5213109482382285, 'n_estimators': 158, 'learning_rate': 0.054487916689644846}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.24708172105946397
Fold 2 IBS: 0.21005062605084635
Fold 3 IBS: 0.23658239945020892
Fold 4 IBS: 0.2291089640988238
Fold 5 IBS: 0.2105403389046395
[I 2024-04-14 23:55:51,728] Trial 97 finished with value: 0.2266728099127965 and parameters: {'subsample': 0.5389995281019297, 'dropout_rate': 0.6077476353900257, 'n_estimators': 73, 'learning_rate': 0.014931416501109941}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.26183286577938214
Fold 2 IBS: 0.20183342233582774
Fold 3 IBS: 0.2575253907596287
Fold 4 IBS: 0.23417362585495322
Fold 5 IBS: 0.21506795896798186
[I 2024-04

In [143]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [144]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.659
train_ibs:  0.227


#### Test

In [145]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [146]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10707700162921632,
                                              learning_rate=0.09635955935176935,
                                              n_estimators=311,
                                              random_state=123,
                                              subsample=0.11211713718471546)

C-index score: 0.536


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6341528845468055,
                                              learning_rate=0.014291926646619518,
                                              n_estimators=76, random_state=123,
                                              subsample=0.6070545538128025)

IBS: 0.234


In [147]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [148]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.830,1.0
ExtraSurvivalTrees,0.764,2.0
GradientBoosting,0.713,3.0
CoxLasso,0.691,4.5
CoxElastic,0.691,4.5
CoxPH,0.689,6.0
CoxRidge,0.663,7.0
ComponentwiseGradientBoosting,0.659,8.0


In [149]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.202,1.0
CoxElastic,0.206,2.0
CoxPH,0.207,4.0
CoxLasso,0.207,4.0
ExtraSurvivalTrees,0.207,4.0
ComponentwiseGradientBoosting,0.227,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [150]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.544,1.0
ExtraSurvivalTrees,0.543,2.0
CoxRidge,0.539,3.0
ComponentwiseGradientBoosting,0.536,4.0
CoxElastic,0.533,5.0
CoxPH,0.532,7.0
CoxLasso,0.532,7.0
GradientBoosting,0.532,7.0


In [151]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ComponentwiseGradientBoosting,0.234,3.0
Randomsurvivalforest,0.247,4.0
ExtraSurvivalTrees,0.256,5.0
CoxLasso,0.288,6.5
CoxElastic,0.288,6.5
CoxPH,0.290,8.0


In [154]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/standard/rent/'  # Folder path where you want to save the files

# List of corresponding file namess
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_standard_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [155]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
